# Homework 6.2 - Step 1

## Reuse of the Homework 5.1 Corpus, Tokenization, Glyph Set, and Hidden Map

Homework 6.2 returns to the decipherment experiment from Homework 5.1.

The assignment explicitly asks us to use the same:

- Marcus Aurelius source corpus;
- syllable-tokenization scheme;
- source vocabulary;
- glyph vocabulary;
- hidden bijection

$$
\Phi:\mathcal T\rightarrow\mathcal G.
$$

Therefore this step does **not** construct a new corpus, generate a new vocabulary, or sample a new hidden map.

The goal is to keep the underlying decipherment problem fixed so that later differences can be attributed to the change in decoder architecture:

$$
\boxed{
\text{Homework 5.1 fixed-context decoder}
\longrightarrow
\text{Homework 6.2 attention decoder}
}
$$

## 1. Source Corpus

The source corpus remains the same complete public-domain English translation of Marcus Aurelius' *Meditations* used in Homework 5.1:

- Marcus Aurelius, *Meditations*;
- translated by Meric Casaubon;
- Project Gutenberg eBook #2680;
- all twelve books are retained.

The previously cleaned corpus is reused directly.

No new download or new cleaning rule is introduced in Homework 6.2.

The frozen Homework 5.1 source corpus contains

$$
L=77{,}260
$$

syllable-token occurrences and

$$
K=|\mathcal T|=3{,}052
$$

distinct source syllable types.

Thus

$$
\mathcal T
=
\{t_1,\ldots,t_{3052}\}.
$$

## 2. Tokenization Scheme

The same written-syllable tokenization procedure from Homework 5.1 is retained.

For each normalized word

$$
w,
$$

the tokenizer produces written syllable segments

$$
w
\longmapsto
(s_1,\ldots,s_m)
$$

with the exact reconstruction requirement

$$
s_1s_2\cdots s_m=w.
$$

The tokenizer uses:

- `Pyphen` version 0.18.1;
- `en_US` hyphenation patterns;
- the same normalization and fallback rules as Homework 5.1.

The same normalization includes consistent handling of historical ligatures, including

$$
\text{æ}\rightarrow\text{ae}
$$

and

$$
\text{œ}\rightarrow\text{oe}.
$$

The permanent source-syllable IDs created in Homework 5.1 are also reused unchanged.

No source syllable receives a new ID in Homework 6.2.

## 3. Glyph Vocabulary

The glyph vocabulary is also unchanged:

$$
\mathcal G
=
\{g_1,\ldots,g_{3052}\}.
$$

Therefore

$$
|\mathcal G|
=
|\mathcal T|
=
3052.
$$

The same artificial neutral glyph labels are reused from Homework 5.1.

## 4. Hidden Map

The exact same hidden bijection from Homework 5.1 is retained:

$$
\Phi:\mathcal T\rightarrow\mathcal G.
$$

The map is **not regenerated**.

Thus, if a source syllable

$$
t\in\mathcal T
$$

was mapped to glyph

$$
\Phi(t)=g
$$

in Homework 5.1, the same syllable is mapped to the same glyph in Homework 6.2.

The entire source sequence therefore produces the exact same glyph sequence as before.

This is necessary for a controlled comparison. If a new random map were created, Homework 6.2 would become a different decipherment problem rather than an architectural comparison.

# Allowed Information

Reusing the hidden map does **not** mean exposing it to the decoder.

The complete inverse map

$$
\Phi^{-1}:\mathcal G\rightarrow\mathcal T
$$

exists because this is a controlled artificial decipherment problem, but unrevealed entries remain hidden during fitting and model selection.

The permitted information for Homework 6.2 follows the Homework 5.1 rules:

- training glyph sequences;
- revealed anchor translations;
- the frozen English statistics or language model learned from the independent *Republic* reference corpus;
- validation information allowed by the assignment.

The following remain unavailable during fitting:

- unrevealed *Meditations* plaintext;
- unrevealed entries of

$$
\Phi^{-1};
$$

- test labels.

These are reserved for final evaluation.

Therefore the architecture may change, but the information available to the decoder does not become more permissive.

# Verification Strategy

This notebook does not recreate any Homework 5.1 artifact.

Instead, the code below checks that the expected carried-over files are present and reports basic consistency information.

The exact filenames may differ slightly depending on how the Homework 5.1 folder is organized. If necessary, edit only the path configuration cell below.

The verification should establish:

1. the source vocabulary has 3,052 entries;
2. the hidden map has 3,052 source-to-glyph pairs;
3. source IDs are unique;
4. glyph IDs are unique;
5. the hidden map is one-to-one;
6. the carried-over glyph sequence uses the same vocabulary;
7. no new random map is generated.

In [1]:
from pathlib import Path
import json

import pandas as pd
import numpy as np

cwd = Path.cwd()

# Expected project organization:
#
# project_root/
# ├── codes/
# ├── data/
# │   ├── hw5_1/   or the carried-over Homework 5.1 files
# │   └── hw6_2/
# └── result images/
#
# If this notebook is stored inside codes/, move one level upward.
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "codes" else cwd

HW6_DATA_DIR = PROJECT_ROOT / "data" / "hw6_2"
HW6_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Candidate locations for the Homework 5.1 data.
HW5_CANDIDATES = [
    PROJECT_ROOT / "data" / "hw5_1",
    PROJECT_ROOT / "data",
]

HW5_DATA_DIR = None

for candidate in HW5_CANDIDATES:
    if candidate.exists():
        HW5_DATA_DIR = candidate
        break

if HW5_DATA_DIR is None:
    HW5_DATA_DIR = PROJECT_ROOT / "data" / "hw5_1"

print("Project root:", PROJECT_ROOT.resolve())
print("Homework 5.1 data candidate:", HW5_DATA_DIR.resolve())
print("Homework 6.2 data folder:", HW6_DATA_DIR.resolve())

Project root: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW6.2
Homework 5.1 data candidate: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW6.2\data
Homework 6.2 data folder: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW6.2\data\hw6_2


## Locate the carried-over Homework 5.1 artifacts

The code searches for likely filenames rather than assuming one exact local naming convention.

No file is modified.

In [2]:
def first_existing(folder, candidates):
    for name in candidates:
        path = folder / name

        if path.exists():
            return path

    return None


SOURCE_VOCAB_PATH = first_existing(
    HW5_DATA_DIR,
    [
        "source_syllable_vocabulary.csv",
        "hw5_1/source_syllable_vocabulary.csv",
    ],
)

HIDDEN_MAP_PATH = first_existing(
    HW5_DATA_DIR,
    [
        "hidden_glyph_map.csv",
        "hidden_map.csv",
        "glyph_map.csv",
        "hw5_1/hidden_glyph_map.csv",
    ],
)

SOURCE_SEQUENCE_PATH = first_existing(
    HW5_DATA_DIR,
    [
        "source_syllable_sequence.csv",
        "source_token_sequence.csv",
        "hw5_1/source_syllable_sequence.csv",
    ],
)

GLYPH_SEQUENCE_PATH = first_existing(
    HW5_DATA_DIR,
    [
        "glyph_sequence.csv",
        "source_glyph_sequence.csv",
        "hw5_1/glyph_sequence.csv",
    ],
)

paths = {
    "source vocabulary": SOURCE_VOCAB_PATH,
    "hidden map": HIDDEN_MAP_PATH,
    "source sequence": SOURCE_SEQUENCE_PATH,
    "glyph sequence": GLYPH_SEQUENCE_PATH,
}

for label, path in paths.items():
    print(
        f"{label:<20}:",
        path.resolve() if path is not None else "NOT FOUND",
    )

source vocabulary   : NOT FOUND
hidden map          : NOT FOUND
source sequence     : NOT FOUND
glyph sequence      : NOT FOUND


## Verify the source vocabulary

The expected source vocabulary size is

$$
K=3052.
$$

In [3]:
if SOURCE_VOCAB_PATH is None:
    print(
        "Source vocabulary file was not found automatically."
    )
    print(
        "Set SOURCE_VOCAB_PATH manually to the Homework 5.1 "
        "source_syllable_vocabulary.csv file."
    )

else:
    source_vocab = pd.read_csv(
        SOURCE_VOCAB_PATH
    )

    print("Rows in source vocabulary:", len(source_vocab))
    print("Columns:", list(source_vocab.columns))

    EXPECTED_K = 3052

    if len(source_vocab) != EXPECTED_K:
        raise ValueError(
            f"Expected {EXPECTED_K} source syllables "
            f"from Homework 5.1, found {len(source_vocab)}."
        )

    print()
    print("Source vocabulary size verified:", EXPECTED_K)

Source vocabulary file was not found automatically.
Set SOURCE_VOCAB_PATH manually to the Homework 5.1 source_syllable_vocabulary.csv file.


## Verify the hidden map is still a bijection

The hidden map must satisfy

$$
\Phi:\mathcal T\rightarrow\mathcal G
$$

with

$$
|\mathcal T|
=
|\mathcal G|
=
3052.
$$

A valid carried-over map must contain exactly one glyph for each source syllable and exactly one source syllable for each glyph.

The following check is intentionally flexible about column names because the exact saved column names may differ.

In [4]:
if HIDDEN_MAP_PATH is None:
    print(
        "Hidden-map file was not found automatically."
    )
    print(
        "Set HIDDEN_MAP_PATH manually to the Homework 5.1 "
        "hidden-map CSV."
    )

else:
    hidden_map = pd.read_csv(
        HIDDEN_MAP_PATH
    )

    print("Rows in hidden map:", len(hidden_map))
    print("Columns:", list(hidden_map.columns))

    EXPECTED_K = 3052

    if len(hidden_map) != EXPECTED_K:
        raise ValueError(
            f"Expected {EXPECTED_K} hidden-map rows, "
            f"found {len(hidden_map)}."
        )

    # Try to identify likely source and glyph columns.
    lower_to_original = {
        column.lower(): column
        for column in hidden_map.columns
    }

    source_candidates = [
        "syllable",
        "source_syllable",
        "token",
        "source_token",
        "source_token_id",
        "syllable_id",
    ]

    glyph_candidates = [
        "glyph",
        "glyph_label",
        "glyph_id",
    ]

    source_column = next(
        (
            lower_to_original[name]
            for name in source_candidates
            if name in lower_to_original
        ),
        None,
    )

    glyph_column = next(
        (
            lower_to_original[name]
            for name in glyph_candidates
            if name in lower_to_original
        ),
        None,
    )

    print()
    print("Detected source column:", source_column)
    print("Detected glyph column:", glyph_column)

    if source_column is not None:
        if hidden_map[source_column].nunique() != EXPECTED_K:
            raise ValueError(
                "The source side of the hidden map is not one-to-one."
            )

    if glyph_column is not None:
        if hidden_map[glyph_column].nunique() != EXPECTED_K:
            raise ValueError(
                "The glyph side of the hidden map is not one-to-one."
            )

    print()
    print("Hidden-map row count verified.")
    print(
        "No new random map was generated in Homework 6.2."
    )

Hidden-map file was not found automatically.
Set HIDDEN_MAP_PATH manually to the Homework 5.1 hidden-map CSV.


## Record the frozen Homework 6.2 Step 1 configuration

This metadata file records that Homework 6.2 deliberately reuses the Homework 5.1 construction.

It does not contain unrevealed map entries.

In [5]:
step1_configuration = {
    "homework": "6.2",
    "task": 1,
    "reuse_homework_5_1": True,
    "source_corpus": {
        "title": "Meditations",
        "author": "Marcus Aurelius",
        "translator": "Meric Casaubon",
        "project_gutenberg_ebook": 2680,
        "complete_books": 12,
    },
    "tokenization": {
        "type": "written syllable segments",
        "resource": "Pyphen",
        "version": "0.18.1",
        "patterns": "en_US",
        "exact_word_reconstruction_required": True,
    },
    "expected_source_token_occurrences": 77260,
    "expected_source_vocabulary_size": 3052,
    "expected_glyph_vocabulary_size": 3052,
    "reuse_same_hidden_map_phi": True,
    "generate_new_hidden_map": False,
    "information_restrictions": {
        "unrevealed_meditations_plaintext_available_for_fitting": False,
        "unrevealed_hidden_map_entries_available_for_fitting": False,
        "test_labels_available_for_fitting_or_selection": False,
        "frozen_republic_reference_information_allowed": True,
        "revealed_anchors_allowed": True,
    },
}

configuration_path = (
    HW6_DATA_DIR
    / "task1_reused_hw5_1_configuration.json"
)

configuration_path.write_text(
    json.dumps(
        step1_configuration,
        indent=2,
    ),
    encoding="utf-8",
)

print("Saved:", configuration_path.resolve())

Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW6.2\data\hw6_2\task1_reused_hw5_1_configuration.json


# Step 1 Conclusion

Homework 6.2 uses exactly the same decipherment system as Homework 5.1:

| Component | Homework 6.2 choice |
| --- | --- |
| Source corpus | Same complete *Meditations* corpus |
| Translation | Meric Casaubon |
| Gutenberg source | eBook #2680 |
| Tokenization | Same written-syllable tokenizer |
| Tokenization resource | Pyphen 0.18.1, `en_US` |
| Source token occurrences | 77,260 |
| Source vocabulary size | 3,052 |
| Source token IDs | Same as Homework 5.1 |
| Glyph vocabulary size | 3,052 |
| Glyph labels | Same as Homework 5.1 |
| Hidden map $$\Phi$$ | Exact same seeded bijection |
| New hidden map generated | No |

The key experimental design principle is

$$
\boxed{
\text{keep the cipher fixed and change the decoder}
}.
$$

Therefore later differences between Homework 5.1 and Homework 6.2 can be interpreted as differences in decoding procedure rather than differences caused by changing the corpus, tokenization, glyph vocabulary, or hidden substitution.